#  Install and Import

In [ ]:
!pip install pyspark==3.5.1 -q

import os
import ast
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, StructType, StructField, DoubleType
)
from pyspark.sql.functions import udf

from pyspark.ml.feature import (
    VectorAssembler, StringIndexer, StandardScaler
)
from pyspark.ml.classification import (
    DecisionTreeClassifier,
    RandomForestClassifier,
    LogisticRegression,
)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("All imports done.")

All imports done.


#  Start Spark

In [ ]:
try:
    spark.stop()
except Exception:
    pass

os.environ["PYSPARK_PYTHON"]        = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"

spark = (SparkSession.builder
    .appName("EduLink_CareerFit_V3")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.python.worker.reuse", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark started successfully.")

Spark version: 3.5.1
Spark started successfully.


# Load Dataset

In [ ]:
DATA_PATH = "synthetic_exam_master_2000.csv"

spark_df_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH))

print("Dataset shape:")
print("  Rows:   ", spark_df_raw.count())
print("  Columns:", len(spark_df_raw.columns))
spark_df_raw.printSchema()
spark_df_raw.show(3, truncate=False)

Dataset shape:
  Rows:    2000
  Columns: 99
root
 |-- student_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- school: string (nullable = true)
 |-- district: string (nullable = true)
 |-- level: string (nullable = true)
 |-- stream: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- location: string (nullable = true)
 |-- time_horizon: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- preferred_language: string (nullable = true)
 |-- device_access: string (nullable = true)
 |-- english_level: string (nullable = true)
 |-- scholarship_need: string (nullable = true)
 |-- ol_math: string (nullable = true)
 |-- ol_science: string (nullable = true)
 |-- ol_ict: string (nullable = true)
 |-- Q1: integer (nullable = true)
 |-- Q2: integer (nullable = true)
 |-- Q3: integer (nullable = true)
 |-- Q4: integer (nullable = true)
 |-- Q5: integer (nullable = true)
 |-- Q6: integer (nullable = true)
 |-- 

# Define Features and Target

In [ ]:
FEATURES = [
    "Analytical_Thinking",
    "Structure_Preference",
    "Creativity_Index",
    "Emotional_Stability",
    "Strategic_Vision",
    "Innovation_Drive",
    "Social_Intelligence",
    "Data_Literacy",
    "Tech_Adaptability",
    "Communication_Skill",
    "Technical_ProblemSolving",
    "Leadership_Capability",
    "Process_Optimization",
    "Tech_Interest",
    "Business_Economics_Interest",
    "Social_Impact_Motivation",
    "Future_Orientation",
    "Career_Growth_Mindset",
    "Entrepreneurship_Orientation",
    "Global_Innovation_Alignment",
    "R", "I", "A", "S", "E", "C",
]

TARGET = "career_cluster"

missing = [c for c in FEATURES + [TARGET]
           if c not in spark_df_raw.columns]

if missing:
    raise ValueError(
        f"Missing columns: {missing}. "
        f"Run Mathematical Layer notebook first."
    )

print(f"Features:  {len(FEATURES)} (20 composites + 6 RIASEC)")
print(f"Target:    {TARGET}")
print()
print("Cluster distribution:")
spark_df_raw.groupBy(TARGET).count()\
    .orderBy(F.col("count").desc()).show()

Features:  26 (20 composites + 6 RIASEC)
Target:    career_cluster

Cluster distribution:
+--------------------+-----+
|      career_cluster|count|
+--------------------+-----+
|Digital_Marketing...|  891|
|Network_Infrastru...|  455|
| Data_AI_Engineering|  350|
|    UX_Creative_Tech|  123|
|    IT_Operations_QA|  106|
|Business_IT_Manag...|   64|
|Software_Web_Engi...|   11|
+--------------------+-----+



#Cast Feature Columns

In [ ]:
df_ml = spark_df_raw

for col in FEATURES:
    df_ml = df_ml.withColumn(
        col,
        F.col(col).cast(DoubleType())
    )

df_ml = df_ml.fillna(0.0, subset=FEATURES)
df_ml = df_ml.filter(F.col(TARGET).isNotNull())
df_ml = df_ml.withColumn(
    "student_id",
    F.col("student_id").cast(StringType())
)

print("Features cast to DoubleType.")
print("Rows after filtering nulls:", df_ml.count())

Features cast to DoubleType.
Rows after filtering nulls: 2000


# Label Encoding

In [ ]:
label_indexer = StringIndexer(
    inputCol=TARGET,
    outputCol="label",
    handleInvalid="keep"
)

label_model = label_indexer.fit(df_ml)
df_ml       = label_model.transform(df_ml)
labels      = label_model.labels

print("Career cluster labels:")
for i, lbl in enumerate(labels):
    print(f"  {i} → {lbl}")

Career cluster labels:
  0 → Digital_Marketing_Media
  1 → Network_Infrastructure
  2 → Data_AI_Engineering
  3 → UX_Creative_Tech
  4 → IT_Operations_QA
  5 → Business_IT_Management
  6 → Software_Web_Engineering


# Feature Assembly and Scaling

In [ ]:
assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol="raw_features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

df_ml        = assembler.transform(df_ml)
scaler_model = scaler.fit(df_ml)
df_ml        = scaler_model.transform(df_ml)

print("Feature vector assembled and scaled.")
print("Feature vector size:", len(FEATURES))

Feature vector assembled and scaled.
Feature vector size: 26


# Train Test Split

In [ ]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Test rows: ", test_df.count())
print()

# Check cluster balance in train set
print("Train cluster distribution:")
train_df.groupBy(TARGET).count()\
    .orderBy(F.col("count").desc()).show()

Train rows: 1642
Test rows:  358

Train cluster distribution:
+--------------------+-----+
|      career_cluster|count|
+--------------------+-----+
|Digital_Marketing...|  730|
|Network_Infrastru...|  366|
| Data_AI_Engineering|  286|
|    UX_Creative_Tech|  109|
|    IT_Operations_QA|   91|
|Business_IT_Manag...|   51|
|Software_Web_Engi...|    9|
+--------------------+-----+



#  Train and Evaluate All Models

In [ ]:
accuracy_eval  = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction",
    metricName="accuracy")
f1_eval        = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction",
    metricName="f1")
precision_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction",
    metricName="weightedPrecision")
recall_eval    = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction",
    metricName="weightedRecall")

models = {
    "DecisionTree": DecisionTreeClassifier(
        featuresCol="features", labelCol="label",
        maxDepth=8, seed=42
    ),
    "RandomForest": RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=100, maxDepth=10, seed=42
    ),
    "LogisticRegression": LogisticRegression(
        featuresCol="features", labelCol="label",
        maxIter=200
    ),
}

trained_models = {}
results        = []

for name, model in models.items():
    print(f"Training {name}...")
    trained   = model.fit(train_df)
    preds     = trained.transform(test_df)
    acc       = accuracy_eval.evaluate(preds)
    f1        = f1_eval.evaluate(preds)
    precision = precision_eval.evaluate(preds)
    recall    = recall_eval.evaluate(preds)

    trained_models[name] = trained
    results.append({
        "Model":              name,
        "Accuracy":           round(acc,       4),
        "F1":                 round(f1,        4),
        "Weighted_Precision": round(precision, 4),
        "Weighted_Recall":    round(recall,    4),
    })
    print(f"  Accuracy={acc:.4f} F1={f1:.4f} "
          f"Precision={precision:.4f} Recall={recall:.4f}")

print()
print("=== MODEL COMPARISON ===")
results_spark = spark.createDataFrame(results)
results_spark.orderBy(F.col("F1").desc()).show()

Training DecisionTree...
  Accuracy=0.6788 F1=0.6717 Precision=0.6654 Recall=0.6788
Training RandomForest...
  Accuracy=0.7709 F1=0.7491 Precision=0.7748 Recall=0.7709
Training LogisticRegression...
  Accuracy=0.9665 F1=0.9706 Precision=0.9761 Recall=0.9665

=== MODEL COMPARISON ===
+--------+------+------------------+------------------+---------------+
|Accuracy|    F1|             Model|Weighted_Precision|Weighted_Recall|
+--------+------+------------------+------------------+---------------+
|  0.9665|0.9706|LogisticRegression|            0.9761|         0.9665|
|  0.7709|0.7491|      RandomForest|            0.7748|         0.7709|
|  0.6788|0.6717|      DecisionTree|            0.6654|         0.6788|
+--------+------+------------------+------------------+---------------+



# Select and Save Best Model

In [ ]:
import pandas as pd

results_pdf  = pd.DataFrame(results).sort_values("F1", ascending=False)
best_name    = results_pdf.iloc[0]["Model"]
best_model   = trained_models[best_name]

print("Best model:", best_name)
print("Best F1:   ", results_pdf.iloc[0]["F1"])
print("Best Acc:  ", results_pdf.iloc[0]["Accuracy"])

best_model.write().overwrite().save("career_fit_model_v3")
label_model.write().overwrite().save("career_fit_label_model_v3")

results_pdf.to_csv("model_evaluation_results.csv", index=False)

print()
print("Saved: career_fit_model_v3/")
print("Saved: career_fit_label_model_v3/")
print("Saved: model_evaluation_results.csv")

Best model: LogisticRegression
Best F1:    0.9706
Best Acc:   0.9665

Saved: career_fit_model_v3/
Saved: career_fit_label_model_v3/
Saved: model_evaluation_results.csv


#  Feature Importances (Random Forest)

In [ ]:
if best_name == "RandomForest":
    importances = best_model.featureImportances.toArray()
    imp_pdf     = pd.DataFrame({
        "feature":    FEATURES,
        "importance": importances
    }).sort_values("importance", ascending=False)

    print("=== TOP 10 FEATURE IMPORTANCES ===")
    print(imp_pdf.head(10).to_string(index=False))

    imp_spark = spark.createDataFrame(imp_pdf)
    imp_spark.orderBy(F.col("importance").desc()).show()

    imp_pdf.to_csv("feature_importances.csv", index=False)
    print("Saved: feature_importances.csv")
else:
    print(f"Best is {best_name} — no feature importances")

Best is LogisticRegression — no feature importances


# New Section

# Generate Predictions with Top 3 Clusters\\\

In [ ]:
predictions = best_model.transform(df_ml)

# UDF to extract top 3 clusters from probability vector
def extract_top3(probability_vector):
    probs    = list(probability_vector)
    top3_idx = sorted(
        range(len(probs)),
        key=lambda i: probs[i],
        reverse=True
    )[:3]
    return str((
        labels[top3_idx[0]],
        labels[top3_idx[1]],
        labels[top3_idx[2]]
    ))

extract_top3_udf = udf(extract_top3, StringType())

predictions = predictions.withColumn(
    "top3_clusters",
    extract_top3_udf(F.col("probability"))
)

print("Predictions generated for", predictions.count(), "students")
predictions.select(
    "student_id", TARGET, "top3_clusters"
).show(5, truncate=False)

Predictions generated for 2000 students
+----------+-----------------------+-------------------------------------------------------------------------------+
|student_id|career_cluster         |top3_clusters                                                                  |
+----------+-----------------------+-------------------------------------------------------------------------------+
|1         |IT_Operations_QA       |('IT_Operations_QA', 'Digital_Marketing_Media', 'Network_Infrastructure')      |
|2         |Network_Infrastructure |('Network_Infrastructure', 'Data_AI_Engineering', 'IT_Operations_QA')          |
|3         |Digital_Marketing_Media|('Digital_Marketing_Media', 'Network_Infrastructure', 'Data_AI_Engineering')   |
|4         |Digital_Marketing_Media|('Digital_Marketing_Media', 'Business_IT_Management', 'Network_Infrastructure')|
|5         |Digital_Marketing_Media|('Digital_Marketing_Media', 'Network_Infrastructure', 'Data_AI_Engineering')   |
+----------+------------

#  Extract Top1, Top2, Top3 as Separate Columns

In [ ]:
# ── FIX 1: Split top3_clusters into separate columns ──────
# Other notebooks (Education, Salary, AI Reasoning) need:
#   Top1, Top2, Top3 as individual string columns

def get_top1(s):
    return ast.literal_eval(s)[0]

def get_top2(s):
    return ast.literal_eval(s)[1]

def get_top3(s):
    return ast.literal_eval(s)[2]

top1_udf = udf(get_top1, StringType())
top2_udf = udf(get_top2, StringType())
top3_udf = udf(get_top3, StringType())

predictions = predictions\
    .withColumn("Top1", top1_udf(F.col("top3_clusters")))\
    .withColumn("Top2", top2_udf(F.col("top3_clusters")))\
    .withColumn("Top3", top3_udf(F.col("top3_clusters")))

print("Top1, Top2, Top3 columns added.")
predictions.select(
    "student_id", "Top1", "Top2", "Top3"
).show(5, truncate=False)

Top1, Top2, Top3 columns added.
+----------+-----------------------+-----------------------+----------------------+
|student_id|Top1                   |Top2                   |Top3                  |
+----------+-----------------------+-----------------------+----------------------+
|1         |IT_Operations_QA       |Digital_Marketing_Media|Network_Infrastructure|
|2         |Network_Infrastructure |Data_AI_Engineering    |IT_Operations_QA      |
|3         |Digital_Marketing_Media|Network_Infrastructure |Data_AI_Engineering   |
|4         |Digital_Marketing_Media|Business_IT_Management |Network_Infrastructure|
|5         |Digital_Marketing_Media|Network_Infrastructure |Data_AI_Engineering   |
+----------+-----------------------+-----------------------+----------------------+
only showing top 5 rows



# Define Cluster Roles

In [ ]:
CLUSTER_ROLES = {
    "Data_AI_Engineering": [
        "Data Scientist", "ML Engineer", "Data Engineer",
        "AI Developer", "Database Administrator",
        "Database Designer", "Computer Database Assistant"
    ],
    "Software_Web_Engineering": [
        "Software Developer", "Software Engineer",
        "Computer Programmer", "Web Developer",
        "Programmer Analyst", "Applications Programmer",
        "Systems Programmer", "Webmaster",
        "Computer Games Developer"
    ],
    "Network_Infrastructure": [
        "Network Engineer", "Network Administrator",
        "Systems Administrator", "IT Infrastructure Engineer",
        "Infrastructure (IT) Architect",
        "Telecommunications Engineer",
        "Network Architect", "Network Manager"
    ],
    "IT_Operations_QA": [
        "IT Quality Assurance Specialist", "IT Auditor",
        "Software Tester", "QA Engineer",
        "IT Support Engineer", "IT Operations Analyst",
        "Site Reliability Associate", "Helpdesk Analyst",
        "Systems Support Officer"
    ],
    "UX_Creative_Tech": [
        "UI/UX Designer", "Product Designer",
        "Graphic Designer", "Creative Technologist",
        "Interaction Designer", "Motion Designer",
        "Front-End Designer", "Multimedia Designer",
        "UX Researcher"
    ],
    "Business_IT_Management": [
        "Business Analyst", "IT Project Manager",
        "Product Manager", "IT Manager", "Scrum Master",
        "IT Consultant", "Business Systems Analyst",
        "Operations Manager", "Strategy Associate"
    ],
    "Digital_Marketing_Media": [
        "Digital Marketer", "SEO Specialist",
        "Content Strategist", "Social Media Manager",
        "Growth Marketer", "Brand Executive",
        "Marketing Analyst", "Media Planner",
        "Campaign Manager"
    ],
    "Hardware_Systems": [
        "Embedded Systems Engineer", "Hardware Engineer",
        "IoT Engineer", "Systems Engineer", "Field Engineer",
        "Electronics Engineer", "Mechatronics Associate",
        "Device Technician", "Lab Engineer"
    ],
}

print("CLUSTER_ROLES defined for 8 clusters.")
for k, v in CLUSTER_ROLES.items():
    print(f"  {k}: {len(v)} roles")

CLUSTER_ROLES defined for 8 clusters.
  Data_AI_Engineering: 7 roles
  Software_Web_Engineering: 9 roles
  Network_Infrastructure: 8 roles
  IT_Operations_QA: 9 roles
  UX_Creative_Tech: 9 roles
  Business_IT_Management: 9 roles
  Digital_Marketing_Media: 9 roles
  Hardware_Systems: 9 roles


# Generate role_1 to role_10 Columns

In [ ]:
# ── FIX 2: Generate role_1 to role_10 as separate columns ──
# Job Recommendation and Salary notebooks need:
#   role_1, role_2 ... role_10 as individual columns

def generate_roles_str(cluster_tuple_string):
    """
    From top3_clusters tuple string generate
    10 role names as pipe-separated string.
    Top1 → 5 roles, Top2 → 3 roles, Top3 → 2 roles
    """
    clusters = ast.literal_eval(cluster_tuple_string)
    top1, top2, top3 = clusters
    roles = []
    roles.extend(CLUSTER_ROLES.get(top1, [])[:5])
    roles.extend(CLUSTER_ROLES.get(top2, [])[:3])
    roles.extend(CLUSTER_ROLES.get(top3, [])[:2])
    seen, unique = set(), []
    for r in roles:
        if r not in seen:
            seen.add(r)
            unique.append(r)
    return "|".join(unique[:10])

roles_udf = udf(generate_roles_str, StringType())

predictions = predictions.withColumn(
    "all_roles",
    roles_udf(F.col("top3_clusters"))
)

# Split pipe-separated roles into individual columns
for i in range(1, 11):
    predictions = predictions.withColumn(
        f"role_{i}",
        F.split(F.col("all_roles"), "\\|").getItem(i - 1)
    )

print("role_1 to role_10 columns added.")
predictions.select(
    "student_id",
    *[f"role_{i}" for i in range(1, 6)]
).show(5, truncate=False)

role_1 to role_10 columns added.
+----------+-------------------------------+---------------------+---------------------+--------------------------+-----------------------------+
|student_id|role_1                         |role_2               |role_3               |role_4                    |role_5                       |
+----------+-------------------------------+---------------------+---------------------+--------------------------+-----------------------------+
|1         |IT Quality Assurance Specialist|IT Auditor           |Software Tester      |QA Engineer               |IT Support Engineer          |
|2         |Network Engineer               |Network Administrator|Systems Administrator|IT Infrastructure Engineer|Infrastructure (IT) Architect|
|3         |Digital Marketer               |SEO Specialist       |Content Strategist   |Social Media Manager      |Growth Marketer              |
|4         |Digital Marketer               |SEO Specialist       |Content Strategist   |Soc

# Add Student_ID Capital Column

In [ ]:
# ── FIX 3: Add Student_ID (capital) column ─────────────────
# AI Reasoning Layer and Education Path need:
#   Student_ID (capital S and capital ID)

predictions = predictions.withColumn(
    "Student_ID",
    F.col("student_id").cast(StringType())
)

print("Student_ID (capital) column added.")
predictions.select(
    "student_id", "Student_ID"
).show(3)

Student_ID (capital) column added.
+----------+----------+
|student_id|Student_ID|
+----------+----------+
|         1|         1|
|         2|         2|
|         3|         3|
+----------+----------+
only showing top 3 rows



# Build Final Report DataFrame


In [ ]:
# ── Final report with ALL required columns ─────────────────
# Selects only needed columns to avoid duplicate error
# Student_ID added after select with withColumn

final_df = predictions.select(
    # student_id — primary key
    F.col("student_id").alias("student_id"),

    # Career cluster label
    F.col(TARGET).alias("actual_career_cluster"),

    # Top 3 clusters as tuple string (backward compatibility)
    F.col("top3_clusters"),

    # Top 3 clusters as separate columns
    # Needed by: Education Path, AI Reasoning, Salary
    F.col("Top1"),
    F.col("Top2"),
    F.col("Top3"),

    # 10 individual role columns
    # Needed by: Job Recommendation, Salary Prediction
    *[F.col(f"role_{i}") for i in range(1, 11)],
)

# Add Student_ID (capital) AFTER select
# to avoid duplicate column error
# Needed by: AI Reasoning Layer, Education Path
final_df = final_df.withColumn(
    "Student_ID",
    F.col("student_id").cast(StringType())
)

print("Final report columns:")
for c in final_df.columns:
    print(f"  {c}")
print()
print("Total rows:", final_df.count())
final_df.show(3, truncate=False)

Final report columns:
  Student_ID
  actual_career_cluster
  top3_clusters
  Top1
  Top2
  Top3
  role_1
  role_2
  role_3
  role_4
  role_5
  role_6
  role_7
  role_8
  role_9
  role_10

Total rows: 2000
+----------+-----------------------+----------------------------------------------------------------------------+-----------------------+-----------------------+----------------------+-------------------------------+---------------------+---------------------+--------------------------+-----------------------------+----------------+---------------------+---------------------+-------------------------------+---------------------+
|Student_ID|actual_career_cluster  |top3_clusters                                                               |Top1                   |Top2                   |Top3                  |role_1                         |role_2               |role_3               |role_4                    |role_5                       |role_6          |role_7               |role_8

# Validate Output

In [ ]:
print("=== OUTPUT VALIDATION ===")
print()

total = final_df.count()
print(f"Total students: {total}")
print()

# Check all required columns exist
required_cols = (
    ["student_id", "Student_ID", "actual_career_cluster",
     "top3_clusters", "Top1", "Top2", "Top3"] +
    [f"role_{i}" for i in range(1, 11)]
)

print("Required column check:")
for col in required_cols:
    exists = col in final_df.columns
    status = "OK" if exists else "MISSING"
    print(f"  {col}: {status}")

print()
print("Top1 cluster distribution:")
final_df.groupBy("Top1").count()\
    .orderBy(F.col("count").desc()).show()

print()
print("Null check on key columns:")
for col in ["Top1", "Top2", "Top3", "role_1"]:
    nulls = final_df.filter(F.col(col).isNull()).count()
    print(f"  {col} nulls: {nulls}")

=== OUTPUT VALIDATION ===

Total students: 2000

Required column check:
  student_id: MISSING
  Student_ID: OK
  actual_career_cluster: OK
  top3_clusters: OK
  Top1: OK
  Top2: OK
  Top3: OK
  role_1: OK
  role_2: OK
  role_3: OK
  role_4: OK
  role_5: OK
  role_6: OK
  role_7: OK
  role_8: OK
  role_9: OK
  role_10: OK

Top1 cluster distribution:
+--------------------+-----+
|                Top1|count|
+--------------------+-----+
|Digital_Marketing...|  887|
|Network_Infrastru...|  455|
| Data_AI_Engineering|  348|
|    UX_Creative_Tech|  124|
|    IT_Operations_QA|  107|
|Business_IT_Manag...|   64|
|Software_Web_Engi...|   15|
+--------------------+-----+


Null check on key columns:
  Top1 nulls: 0
  Top2 nulls: 0
  Top3 nulls: 0
  role_1 nulls: 0


# Save career_fit_report.csv

In [ ]:
import glob
import shutil

# ── Save as career_fit_report.csv ──────────────────────────
# This is the exact filename all other notebooks expect

OUTPUT_FILE = "career_fit_report.csv"

(final_df
    .coalesce(1)
    .write
    .option("header", True)
    .mode("overwrite")
    .csv("career_fit_report_temp"))

parts = glob.glob("career_fit_report_temp/part-*.csv")
if parts:
    shutil.copy(parts[0], OUTPUT_FILE)
    shutil.rmtree("career_fit_report_temp")

print("="*55)
print("SAVED:", OUTPUT_FILE)
print("Rows:", final_df.count())
print()
print("Columns saved:")
for c in final_df.columns:
    print(f"  {c}")
print()
print("This file is ready for:")
print("  - Education Path Recommendation notebook")
print("  - Salary Prediction notebook")
print("  - Job Recommendation notebook")
print("  - AI Reasoning Layer notebook")
print("="*55)

SAVED: career_fit_report.csv
Rows: 2000

Columns saved:
  Student_ID
  actual_career_cluster
  top3_clusters
  Top1
  Top2
  Top3
  role_1
  role_2
  role_3
  role_4
  role_5
  role_6
  role_7
  role_8
  role_9
  role_10

This file is ready for:
  - Education Path Recommendation notebook
  - Salary Prediction notebook
  - Job Recommendation notebook
  - AI Reasoning Layer notebook


#  Final Summary

In [ ]:
print("="*55)
print("  EduLink Career Fit V3 — Complete Summary")
print("="*55)
print()
print(f"  Dataset:         2000 students")
print(f"  Features:        {len(FEATURES)} (20 composites + 6 RIASEC)")
print(f"  Career clusters: 8 Sri Lanka IT clusters")
print(f"  Best model:      {best_name}")
print(f"  Best F1:         {results_pdf.iloc[0]['F1']}")
print(f"  Best Accuracy:   {results_pdf.iloc[0]['Accuracy']}")
print()
print("  Output file: career_fit_report.csv")
print()
print("  Columns in output:")
print("    student_id, Student_ID")
print("    actual_career_cluster")
print("    top3_clusters (tuple string)")
print("    Top1, Top2, Top3 (separate columns)")
print("    role_1 to role_10 (individual role columns)")
print()
print("  Next steps:")
print("    1. Download career_fit_report.csv")
print("    2. Upload to Google Drive FYP1/Dataset/")
print("    3. Run Writing Analysis notebook")
print("    4. Run Job Demand Forecasting notebook")
print("    5. Then run Salary, Education, Job Rec")
print("    6. Finally run AI Reasoning Layer")
print("="*55)

  EduLink Career Fit V3 — Complete Summary

  Dataset:         2000 students
  Features:        26 (20 composites + 6 RIASEC)
  Career clusters: 8 Sri Lanka IT clusters
  Best model:      LogisticRegression
  Best F1:         0.9706
  Best Accuracy:   0.9665

  Output file: career_fit_report.csv

  Columns in output:
    student_id, Student_ID
    actual_career_cluster
    top3_clusters (tuple string)
    Top1, Top2, Top3 (separate columns)
    role_1 to role_10 (individual role columns)

  Next steps:
    1. Download career_fit_report.csv
    2. Upload to Google Drive FYP1/Dataset/
    3. Run Writing Analysis notebook
    4. Run Job Demand Forecasting notebook
    5. Then run Salary, Education, Job Rec
    6. Finally run AI Reasoning Layer
